# PA3 — weights + characteristics, 0CQ single frequency

Holdings-based snapshot from PA Engine as of the most recent calendar quarter end.
Companion to `spar_composite_returns_template.ipynb`, which is returns-based (SPAR).

**Still needed to run:** `PA_DOCUMENT` and the two component names. Account paths, holdings
modes and benchmarks are read off the saved components by Cell 4c.

## FactSet's numbers are precalculated. Nothing here derives them.

PA and SPAR return values already aggregated, compounded and annualised by the engine, at
every grain they publish. This pipeline reshapes and types them for display and does not
recompute any of them. The only arithmetic anywhere is on **dates** (period counts, months of
history) and on **row counts** — neither is a reported number. Where a total from one grain is
printed beside another's, it is labelled as reconciliation, printed only, never written, and
the engine's value still wins.

Two rules follow into the semantic model:

- **Never `SUM` or `AVERAGE` a return across periods in DAX.** Select the precalculated value
  for the grain being displayed. Compounding monthly returns in DAX will disagree with
  FactSet, and FactSet is the number that goes in front of a client.
- **Never sum a weight across grains.** Sector totals come from the sector rows, holdings
  detail from the security rows.

## One multi-port call per tile

All four strategies go through a single unit per tile — `accounts` is a list, so LC, SMID,
LCS and CONC are calculated together. **2 units total.**

`benchmarks` is left **omitted**, so each account uses its own default benchmark as saved.
That is what makes one unit work across four composites with three different benchmarks:
passing an explicit list would raise the question of how PA pairs 4 accounts to 3
benchmarks — counts don't match, so positional pairing is impossible and the schema doesn't
say whether it cross-products. Omitting the field sidesteps it, and each composite's
benchmark is already correct in the document.

The consequence: **the benchmark is not known from config**, so it is captured from the
response instead (Cell 7). Same for the strategy — a shared unit means rows must be
attributed via the output's account column, so `ACCT_TO_CODE` maps it back and the write
asserts every row resolved. `PER_PAIR_UNITS = True` falls back to one unit per port+bench
pair (8 units) where both come from the unit key, which is easier to debug if attribution
misbehaves.

## Weights: `GROUPSALL`, precalculated at every grain

`GROUPSALL` returns group rows **and** security rows in one response, and **PA supplies
precalculated portfolio, benchmark and active weights at every grain.** Those are
authoritative — never re-derive a sector weight by summing its securities. If a sum
disagrees with the sector row, the sector row is right and the difference is PA's
methodology, not an error to correct.

That is also why the grains are landed as **separate tables** (`pa_sector_weights`,
`pa_security_weights`): both carry their own authoritative weights, so anything summing across
a mixed-grain table double-counts.

The portfolio-**total** row is **not landed**. It carries nothing the other two grains lack,
and its one real use — confirming the book adds to 100% — is answered by summing the grains
that do land. The row is still counted and reported, so a component that stops emitting it is
visible rather than silently absent.

### The one place a column sum is meaningful

Weights are it, and only as a **check that the total reaches 100%** — never as a published
figure. Both grains are summed per strategy against a 99–101 band and compared to each other:

- a **security** total short of 100 points at the benchmark-only filter, or at cash being
  dropped for having no portfolio weight
- the **two grains disagreeing** points at the grain classification itself

Everywhere else, FactSet's precalculated value at the grain on display is the number.

Target columns on each holding row:

| Column | Source |
|---|---|
| FSYM perm id / regional id / entity id | component columns, kept separate |
| ticker, security name | component columns |
| **GICS sector** | **derived from the STACH grouping** — see below |
| cash flag | component column, or derived |
| ultimate parent FSYM id | component column, if exposed |
| port / bench / active weight | precalculated by PA at each grain |

## Characteristics: `TOTALS` grain only

Characteristics are published **already aggregated at total grain** — one row per portfolio,
not per sector. So the component runs at `componentdetail="TOTALS"`, there is no grain to
split, and `strategy_code` alone organises the table: **exactly one row per strategy**,
asserted before the write. Two rows for one strategy means either the component is not really
at `TOTALS` or two accounts mapped to the same code.

That makes `pa_characteristics` a one-to-one companion to `dim_strategy` — it needs no grain
filter downstream and goes straight onto a card or a header row.

### GICS sector is a grouping, not a column

Under `GROUPSALL` the security rows are *nested beneath* their sector's group row, so the
sector must be projected down onto each holding to sit in an adjacent column. Cell 7 uses
the grouping column directly when it is already populated on security rows, otherwise
forward-fills from each preceding group row.

> ⚠️ Forward-fill assumes STACH emits rows in hierarchical order. True of PA's grouped
> output, but an assumption about row order rather than a documented guarantee. Cell 7
> reports which path ran and prints holdings-per-sector counts — check one composite
> against the workstation. A mis-fill mislabels every holding without erroring.

### Benchmark-only securities are dropped at security grain

`HIDE_BENCH_ONLY_SECURITIES = True` drops security rows with no portfolio weight — index
constituents not held. Without it a Russell 3000 benchmark contributes thousands of rows
that no top-N holdings view wants.

> The trade-off: those rows carry the security-level **active** weight of names you *don't*
> own, so dropping them means you cannot show largest underweights-not-held from this
> table. Sector-grain active weight is unaffected. Set the flag False if you need them.
> If the component already hides them the filter is a harmless no-op — the count dropped
> is reported either way.

## Dates: PA resolves `0CQ`, and that answer is authoritative

PA has a `DatesApi`; SPAR does not. `convert_pa_dates_to_absolute_format` turns `0CQ` into
a real `YYYYMMDD`, and **that value is what gets sent and what labels every row** — no
locally computed guess. The SPAR notebook calls the same endpoint so both pipelines agree
on one as-of date.

Note `enddate`, `componentid` and `account` are all **required** on that call (only
`startdate` and `calendar` are optional), so it runs *after* component resolution.

## Metadata is captured, not discarded

Every run records the calculation id, `X-DataDirect-Request-Key`,
`X-FactSet-Api-Request-Key`, rate-limit headers, SDK version, resolved dates, and each
component's id / name / path / currency / snapshot flag into `factset.factset_run_log`.
Request keys are what FactSet support needs to pull the exact request, and they are
worthless if not persisted at the time.

## Fee basis does not apply

Weights and characteristics are holdings attributes — no gross/net distinction.

## Versions

| Package | Version |
|---|---|
| `fds.sdk.PAEngine` | **4.0.0** (upstream latest, 2026-07-21) |
| `fds.sdk.utils` | 3.0.1 |
| `fds.protobuf.stach.extensions` | 1.3.3 |
| `deltalake` (delta-rs) | preinstalled — do not pin |

PAEngine 4.0.0 dropped `required` from `PADateParameters.enddate` / `.frequency`, which
**reshuffles positional arguments** across the calculation and dates endpoints
(upstream `BREAKING.md`, 2026-07-21). Every call here passes keywords.

> ⚠️ The PA SDK vendored under `code/python/PAEngine/v3/` in this repo is **2.2.2** and
> predates this. Verify against upstream `main`.

Attach libraries to a **Fabric Environment**, not `%pip`. Interactive first run only:
```
%pip install fds.sdk.PAEngine==4.0.0 fds.sdk.utils==3.0.1 \
             fds.protobuf.stach.extensions==1.3.3
```

In [ ]:
# === Cell 1: credentials ===================================================
%run HBCM_Config

In [ ]:
# === Cell 2: imports + API client ==========================================
import json, time, datetime as dt
import pandas as pd

import fds.sdk.PAEngine
from fds.sdk.PAEngine.api import (
    pa_calculations_api, components_api, accounts_api,
    columns_api, groups_api, frequencies_api, dates_api,
)
from fds.sdk.PAEngine.models import (
    PACalculationParametersRoot, PACalculationParameters,
    PAIdentifier, PADateParameters, CalculationMeta,
)
from deltalake import DeltaTable, write_deltalake
from urllib3 import Retry

SDK_VERSION = fds.sdk.PAEngine.__version__
assert int(SDK_VERSION.split(".")[0]) >= 4, (
    f"fds.sdk.PAEngine {SDK_VERSION} found; this notebook targets >=4.0.0 "
    "(PADateParameters changed shape). Check the bound Fabric Environment."
)
print("PAEngine SDK", SDK_VERSION)

configuration = fds.sdk.PAEngine.Configuration(
    username=FACTSET_USER, password=FACTSET_APIKEY,
)
configuration.retries = Retry(
    total=3, status_forcelist=[500, 502, 503, 504], backoff_factor=2,
    allowed_methods=frozenset(["GET", "POST"]),
)

api_client = fds.sdk.PAEngine.ApiClient(configuration)
calc_api = pa_calculations_api.PACalculationsApi(api_client)
comp_api = components_api.ComponentsApi(api_client)

# --- run metadata ----------------------------------------------------------
# Populated as the notebook proceeds and landed in Cell 9. Request keys are what FactSet
# support needs to retrieve the exact request; they are worthless unless persisted now.
RUN_META = {
    "run_started_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "notebook": "pa_weights_characteristics",
    "engine": "PAEngine",
    "sdk_version": SDK_VERSION,
}

def _field(obj, name):
    """SDK models allow attribute or dict-style access depending on construction."""
    if hasattr(obj, name):
        return getattr(obj, name)
    try:
        return obj.get(name)
    except AttributeError:
        return None

INTERESTING_HEADERS = (
    "X-DataDirect-Request-Key",
    "X-FactSet-Api-Request-Key",
    "X-FactSet-Api-RateLimit-Limit",
    "X-FactSet-Api-RateLimit-Remaining",
    "X-FactSet-Api-RateLimit-Reset",
)

def call_with_headers(fn, *args, **kwargs):
    """Call `fn`'s _with_http_info sibling and return (result, headers).

    The wrapper-returning endpoints don't document their _with_http_info shape, so this
    handles both a 3-tuple and a bare return, and degrades to no headers rather than
    failing the run over telemetry.
    """
    sibling = getattr(fn.__self__, fn.__name__ + "_with_http_info", None)
    if sibling is None:
        return fn(*args, **kwargs), {}
    try:
        out = sibling(*args, **kwargs)
    except TypeError:
        return fn(*args, **kwargs), {}
    if isinstance(out, tuple) and len(out) == 3:
        result, _status, headers = out
        return result, {k: v for k, v in dict(headers or {}).items()
                        if k in INTERESTING_HEADERS}
    return out, {}

# --- STACH parsing, driven by the package's own schema ---------------------
# The API is builder-based: get_row_organized_builder -> set_package -> build ->
# convert_to_dataframe. (There is no get_stach_extension/convert pair — calling that raises
# AttributeError.) `contentorganization` is a row-organized setting here, hence the row
# builder; a Column setting needs get_column_organized_builder.
from fds.protobuf.stach.extensions.StachExtensionFactory import StachExtensionFactory
from fds.protobuf.stach.extensions.StachVersion import StachVersion
from fds.protobuf.stach.v2.RowOrganized_pb2 import RowOrganizedPackage

# STACH declares a type per column, so dates and measures are identified from the package
# rather than sniffed from values or guessed from column names.
STACH_DATE_TYPES = {"date", "datetime", "timestamp"}
STACH_NUMERIC_TYPES = {"double", "float", "int", "integer", "int32", "int64", "long",
                       "decimal", "number", "percent"}

def _row_builder():
    return StachExtensionFactory.get_row_organized_builder(StachVersion.V2)

def _as_package(payload):
    """Parse a result payload into a RowOrganizedPackage, tolerating a wrapper key.

    A wrapper would otherwise yield a package with zero tables and an empty DataFrame —
    silently, which is the worst outcome. Fail with something actionable instead.
    """
    if hasattr(payload, "to_dict"):
        payload = payload.to_dict()
    candidates = [payload]
    if isinstance(payload, dict) and isinstance(payload.get("data"), dict):
        candidates.append(payload["data"])
    for cand in candidates:
        pkg = _row_builder().set_package(cand).get_package()
        if len(pkg.tables):
            return pkg
    raise ValueError(
        "no STACH tables in the response. Confirm meta.format='JsonStach' and a "
        "row-organized contentorganization ('SimplifiedRow' or 'Row')."
    )

def stach_parse(payload):
    """-> list of (table_id, DataFrame, column_schema, group_levels).

    column_schema: {column: {stach_type, is_dimension, is_hidden, format, null_format}}
    group_levels:  per body row, the STACH hierarchy level (0 = outermost group) or None.
                   This is the authoritative grain discriminator for GROUPSALL output — no
                   guessing at a level column or a null security id.

    Each table is converted through its own single-table extension so the DataFrame is
    guaranteed to pair with the table it came from; iterating a protobuf map twice and
    zipping would depend on map ordering, which is unspecified.
    """
    pkg = _as_package(payload)
    out = []
    for tid in list(pkg.tables.keys()):
        table = pkg.tables[tid]
        df = _row_builder().add_table(tid, table).build().convert_to_dataframe()[0]
        # convert_to_dataframe names columns `description or name` — match that exactly.
        schema = {
            (c.description or c.name): {
                "stach_type": (c.type or "").lower(),
                "is_dimension": bool(c.is_dimension),
                "is_hidden": bool(c.is_hidden),
                "format": c.format.format or "",
                "null_format": c.format.null_format or "",
            }
            for c in table.definition.columns
        }
        levels = []
        for r in table.data.rows:
            if RowOrganizedPackage.Row.RowType.Name(r.row_type) == "Header":
                continue
            lv = None
            for _k, detail in r.cell_details.items():
                lv = int(detail.group_level)
                break
            levels.append(lv)
        out.append((tid, df, schema, levels))
    return out

def type_from_schema(df, schema, never_numeric=frozenset()):
    """Type a STACH frame from its declared schema. -> (df, report).

    Three things come from STACH instead of guesswork:
      - the declared `type` decides date vs numeric vs text
      - `is_dimension` protects identifiers, so an FSYM id or a zero-padded code is never
        coerced to a float
      - `null_format` is the exact token meaning "no value" for THAT column, rather than a
        global guess at "--" / "N/A"

    A column with no declared type falls back to sniffing and is reported, so an undeclared
    measure is visible rather than quietly landing as text.
    """
    out = df.copy()
    report = {"date": [], "numeric": [], "string": [], "undeclared": []}
    for c in out.columns:
        meta = schema.get(c, {})
        stype = meta.get("stach_type", "")
        txt = out[c].astype("string").str.strip()
        nullfmt = meta.get("null_format", "")
        drop = {"", "--", "N/A", "NA", "n/a", "None", "nan"} | ({nullfmt} if nullfmt else set())
        txt = txt.where(~txt.isin(drop), pd.NA)

        if c in never_numeric or meta.get("is_dimension"):
            out[c] = txt
            report["string"].append(c)
        elif stype in STACH_DATE_TYPES:
            out[c] = pd.to_datetime(txt, errors="coerce", format="mixed").dt.date
            report["date"].append(c)
        elif stype in STACH_NUMERIC_TYPES:
            out[c] = pd.to_numeric(txt.str.replace(",", "", regex=False).str.rstrip("%"),
                                   errors="coerce").astype("Float64")
            report["numeric"].append(c)
        else:
            report["undeclared"].append(f"{c}:{stype or 'no type'}")
            conv = pd.to_numeric(txt.str.replace(",", "", regex=False).str.rstrip("%"),
                                 errors="coerce")
            nn = txt.notna()
            if not nn.any():
                # All-blank: type numerically so the Delta schema is stable across quarters.
                # Landing text now and a float the quarter it populates cannot be merged.
                out[c] = pd.Series(pd.NA, index=out.index, dtype="Float64")
                report["numeric"].append(c)
            elif (conv.notna() & nn).sum() / nn.sum() >= 0.90:
                out[c] = conv.astype("Float64")
                report["numeric"].append(c)
            else:
                out[c] = txt
                report["string"].append(c)
    return out, report

# --- FactSet numbers are precalculated. Never derive. ----------------------
# PA and SPAR return values already aggregated, compounded and annualised by the engine, at
# every grain they publish. Nothing in this notebook recomputes, sums, compounds or
# annualises a FactSet figure. The only arithmetic performed anywhere is on DATES (period
# counts, months of history) and on ROW COUNTS, neither of which is a reported number.
#
# Where a total is printed next to another grain's total, it is labelled as reconciliation
# and the engine's value still wins. Two rules follow downstream:
#   - never SUM or AVERAGE a return across periods in DAX; select the precalculated value
#     for the grain being displayed (that is what the multi-horizon and cumulative tiles
#     are for)
#   - never sum a weight across grains; use the sector row for sector totals and the
#     security rows for holdings
import re as _re

_HORIZON_RE = _re.compile(
    r"(?P<n>\d+)\s*(?P<unit>y(?:r|ear)?s?|m(?:o|onth)?s?|q(?:tr|uarter)?s?)\b", _re.I)
_ALWAYS_VALID = _re.compile(r"\b(itd|ytd|qtd|mtd|since\s+inception|inception|cumulative)\b",
                            _re.I)

def horizon_months(label):
    """Months of history a horizon label requires. 0 = always valid. None = not a horizon.

    Inception-to-date and the *TD family are always valid: they are defined by whatever
    history exists rather than requiring a fixed span.
    """
    if label is None:
        return None
    text = str(label)
    if _ALWAYS_VALID.search(text):
        return 0
    m = _HORIZON_RE.search(text)
    if not m:
        return None
    n, unit = int(m.group("n")), m.group("unit").lower()
    if unit.startswith("y"):
        return n * 12
    if unit.startswith("q"):
        return n * 3
    return n

def suppress_invalid_horizons(df, months_by_strategy, schema, strategy_col="strategy_code"):
    """Blank horizon COLUMNS a strategy cannot support, and drop ones no strategy can.

    A "5 Year" figure for a composite with one year of history is not an empty cell — the
    engine may return something for it, and it renders as a real number. Suppressing is a
    display decision, not a calculation: no value is altered, only withheld where the window
    does not exist.

    Returns (df, report).
    """
    out = df.copy()
    report = {"blanked": {}, "dropped": [], "horizon_columns": {}}
    if strategy_col not in out.columns or not months_by_strategy:
        report["skipped"] = "no strategy column or no history available"
        return out, report

    for c in list(out.columns):
        meta = schema.get(c, {})
        if meta.get("is_dimension"):
            continue                      # a dimension is never a horizon measure
        need = horizon_months(c)
        if not need:                      # None (not a horizon) or 0 (always valid)
            continue
        report["horizon_columns"][c] = need
        bad = out[strategy_col].map(
            lambda code: need > months_by_strategy.get(code, 0))
        n_bad = int(bad.sum())
        if n_bad == len(out) and len(out):
            out = out.drop(columns=[c])   # no strategy supports it: hide the column
            report["dropped"].append(c)
        elif n_bad:
            out.loc[bad, c] = pd.NA
            report["blanked"][c] = n_bad
    return out, report

def drop_invalid_horizon_rows(df, months_by_strategy, label_col,
                              strategy_col="strategy_code"):
    """Same rule when the component puts horizons in ROWS rather than columns."""
    if label_col not in df.columns or strategy_col not in df.columns:
        return df, {"skipped": "no label or strategy column"}
    need = df[label_col].map(horizon_months)
    have = df[strategy_col].map(lambda c: months_by_strategy.get(c, 0))
    invalid = need.notna() & (need > 0) & (need > have)
    dropped = (df.loc[invalid, [strategy_col, label_col]]
                 .astype("string").agg(" / ".join, axis=1).value_counts().to_dict())
    return df.loc[~invalid].copy(), {"dropped_rows": int(invalid.sum()), "detail": dropped}

def drop_stach_hidden(df, schema):
    """Drop columns STACH itself marks is_hidden — the engine's own display decision."""
    hidden = [c for c in df.columns if schema.get(c, {}).get("is_hidden")]
    return (df.drop(columns=hidden) if hidden else df), hidden

def schema_date_columns(schema):
    """Columns STACH declares as dates — used instead of matching column names."""
    return [c for c, m in schema.items() if m.get("stach_type", "") in STACH_DATE_TYPES]

def dedupe_columns(df, provenance):
    """Rename any STACH column that collides with a provenance column.

    Normalising STACH labels to snake_case can land one on top of an inserted column —
    "Currency" -> currency, for instance. Duplicate names break the Delta write, and
    silently shadow one of the two before that.
    """
    clash = [c for c in df.columns if c in provenance]
    return (df.rename(columns={c: f"{c}_src" for c in clash}), clash) if clash else (df, [])

def table_exists(path):
    """True/False, distinguishing 'no such table' from a real failure.

    A bare try/except around DeltaTable() that falls back to overwrite will destroy every
    prior quarter the first time a transient auth or throttling error shows up.
    """
    try:
        DeltaTable(path)
        return True
    except Exception as e:
        msg = f"{type(e).__name__}: {e}".lower()
        if any(k in msg for k in ("not a delta table", "no log files", "not found",
                                  "does not exist", "notfound", "no such file")):
            return False
        raise RuntimeError(
            f"could not determine whether {path} exists ({e!r}). Refusing to continue: "
            f"treating this as a missing table would overwrite existing history."
        ) from e

# --- strategy_code: the one join key across every PA and SPAR table --------
# The semantic model is filtered to a single strategy at a time, so strategy_code has to be
# present, populated and identically spelled on every table. Declared canonically here and
# asserted on both sides rather than trusted.
STRATEGY_CODES = ("LC", "LCS", "SMID", "CONC")
STRATEGY_LABELS = {
    "LC":   "Large Cap",
    "LCS":  "Large Cap Select",
    "SMID": "SMID",
    "CONC": "Concentrated Equity",
}

def assert_strategy_key(df, table_name, expected=STRATEGY_CODES):
    """strategy_code must be present, fully populated, and in the canonical set.

    A null or off-spec code does not fail loudly downstream — it produces a row that
    silently disappears from every strategy-filtered visual, which is worse than an error.
    """
    assert "strategy_code" in df.columns, f"{table_name}: no strategy_code column"
    s = df["strategy_code"].astype("string")
    n_null = int(s.isna().sum())
    assert n_null == 0, f"{table_name}: {n_null} rows have no strategy_code"

    bad_fmt = sorted(set(s[~s.str.fullmatch(r"[A-Z0-9]{2,4}")].dropna()))
    assert not bad_fmt, (
        f"{table_name}: strategy_code must be 2-4 upper-case alphanumerics; got {bad_fmt}"
    )
    unknown = sorted(set(s.dropna()) - set(expected))
    assert not unknown, (
        f"{table_name}: strategy_code values not in STRATEGY_CODES: {unknown}. "
        f"Either add them to the canonical list or fix the mapping."
    )
    missing = sorted(set(expected) - set(s.dropna()))
    if missing:
        # Not fatal — a tile may legitimately not cover every strategy — but a silently
        # absent strategy looks identical to one with no data.
        print(f"  NOTE {table_name}: no rows for {missing}")
    return sorted(set(s.dropna()))

def assert_unique_grain(df, keys, table_name):
    """The stated grain must actually be unique.

    A duplicate on the declared key is what makes a Power BI relationship fan out and
    weights double — and it shows up as plausible-but-wrong numbers, not as an error.
    """
    present = [k for k in keys if k in df.columns]
    missing = [k for k in keys if k not in df.columns]
    if missing:
        print(f"  NOTE {table_name}: grain columns absent, cannot verify: {missing}")
        return None
    dup = df.duplicated(subset=present, keep=False)
    n = int(dup.sum())
    if n:
        print(f"  *** {table_name}: {n} rows duplicate the declared grain {present}")
        print(df.loc[dup, present].head(12).to_string(index=False))
    assert n == 0, (
        f"{table_name}: {n} rows share a {present} key. Landing this would make any "
        f"relationship on those columns fan out."
    )
    print(f"  OK   {table_name}: unique on {present} ({len(df)} rows)")
    return True

# --- grain classification and hierarchy ------------------------------------
def _blank(s):
    t = s.astype("string").str.strip()
    return t.isna() | (t == "")

def classify_grain(df, id_col="fsym_perm_id", level_col="group_level"):
    """'security' | 'group' | 'total' per row, plus a report.

    FactSet populates the security identifier ONLY at security grain — group and total rows
    leave it blank. So the identifier is the primary signal, and STACH's group_level is the
    tiebreak between group and total: among rows with no identifier, the shallowest level is
    the portfolio total and anything deeper is a group.

    Both signals are needed. An identifier alone cannot separate total from group, and a
    level alone cannot tell a single-level grouping from a total row.
    """
    rep = {}
    has_id = (~_blank(df[id_col])) if id_col in df.columns else pd.Series(False, index=df.index)
    lvl = (pd.to_numeric(df[level_col], errors="coerce")
           if level_col in df.columns else pd.Series(pd.NA, index=df.index))

    grain = pd.Series("group", index=df.index, dtype="object")
    grain[has_id] = "security"

    levels = sorted(set(lvl[~has_id].dropna().tolist()))
    rep["levels_on_non_security_rows"] = levels
    if len(levels) > 1:
        grain[(~has_id) & (lvl == levels[0])] = "total"
    elif len(levels) == 1:
        rep["note"] = (f"only one non-security level ({levels[0]}) — all treated as group; a "
                       f"portfolio total row, if present, is not separable by level")
    else:
        rep["note"] = "no group_level available; non-identifier rows all treated as group"
    rep["counts"] = grain.value_counts().to_dict()
    rep["identifier_on_non_security_rows"] = int((has_id & (grain != "security")).sum())
    return grain, rep

def assign_ancestors(df, level_col="group_level", label_cols=(), prefix="group_l"):
    """The label of each row's nearest ancestor at every shallower level.

    STACH emits rows in hierarchical order, so a stack keyed by level reconstructs the path
    exactly. Strictly better than forward-filling one column: it handles nesting deeper than
    one level and cannot leak a label across a sibling boundary.

    Adds `<prefix>N` per level and `parent_group`. For a sector-grouped weights report the
    security's GICS sector is its ancestor label — derived from the layout, not from a column
    FactSet does not populate at that grain.
    """
    lvl = pd.to_numeric(df[level_col], errors="coerce")
    labels = None
    for c in label_cols:
        if c in df.columns:
            col = df[c].astype("string").str.strip().replace({"": pd.NA})
            labels = col if labels is None else labels.fillna(col)
    if labels is None:
        labels = pd.Series(pd.NA, index=df.index, dtype="string")

    max_lvl = int(lvl.max()) if lvl.notna().any() else 0
    cols = {f"{prefix}{k}": [] for k in range(max_lvl + 1)}
    parents, stack = [], {}
    for idx in df.index:
        li = int(lvl.get(idx)) if pd.notna(lvl.get(idx)) else None
        lab = labels.get(idx)
        if li is not None:
            if pd.notna(lab):
                stack[li] = lab
            for deeper in [k for k in list(stack) if k > li]:
                del stack[deeper]
        for k in range(max_lvl + 1):
            cols[f"{prefix}{k}"].append(stack.get(k, pd.NA)
                                        if (li is None or k < li) else pd.NA)
        anc = [stack[k] for k in sorted(stack) if li is not None and k < li]
        parents.append(anc[-1] if anc else pd.NA)

    out = df.copy()
    for c, vals in cols.items():
        out[c] = pd.array(vals, dtype="string")
    out["parent_group"] = pd.array(parents, dtype="string")
    return out

# --- identifier crosswalk --------------------------------------------------
# What the response calls an account is not necessarily what was sent. PA may echo an Orion
# account id, a `path.ACCT` form, or a display name; benchmarks may come back as a symbol, a
# name, or both. Rather than assume one shape, normalise and record every identifier seen
# against strategy_code, so the semantic model joins on one key and the crosswalk is
# auditable when a vendor changes what it echoes.
def norm_id(v):
    """Comparison form: upper-cased, prefix before ':' dropped, .ACCT/.ACTM suffix dropped."""
    if v is None:
        return None
    t = str(v).strip().upper()
    if ":" in t:
        t = t.rsplit(":", 1)[-1]
    for suf in (".ACCT", ".ACTM", ".OFDB"):
        if t.endswith(suf):
            t = t[: -len(suf)]
    return t.strip("/ ").split("/")[-1] or None

def build_id_index(mapping):
    """{observed form -> strategy_code} for exact, normalised and basename matching."""
    idx = {}
    for raw, code in mapping.items():
        for form in {str(raw).strip(), str(raw).strip().upper(), norm_id(raw)}:
            if form:
                idx.setdefault(form, code)
    return idx

def match_strategy(series, mapping):
    """Map response identifiers to strategy_code. -> (codes, how, unmatched values)."""
    idx = build_id_index(mapping)
    s = series.astype("string").str.strip()
    exact = s.map(lambda v: idx.get(v) if pd.notna(v) else None)
    upper = s.str.upper().map(lambda v: idx.get(v) if pd.notna(v) else None)
    normed = s.map(lambda v: idx.get(norm_id(v)) if pd.notna(v) else None)
    codes = exact.fillna(upper).fillna(normed)
    how = pd.Series(pd.NA, index=s.index, dtype="string")
    how[exact.notna()] = "exact"
    how[exact.isna() & upper.notna()] = "case-insensitive"
    how[exact.isna() & upper.isna() & normed.notna()] = "normalised"
    unmatched = sorted(set(s[codes.isna() & s.notna()].tolist()))
    return codes, how, unmatched

ID_CROSSWALK = []          # accumulated rows -> factset.dim_account_map

def record_identifiers(strategy_code, source, role, sent=None, observed_id=None,
                       observed_name=None, matched_by=None):
    ID_CROSSWALK.append({
        "strategy_code": strategy_code, "source": source, "role": role,
        "sent_value": sent, "observed_id": observed_id, "observed_name": observed_name,
        "matched_by": matched_by,
    })

def write_id_crosswalk(path):
    """Land the crosswalk, replacing this vintage's rows so a re-run does not duplicate."""
    if not ID_CROSSWALK:
        print("no identifiers captured — nothing to write to dim_account_map")
        return None
    df = pd.DataFrame(ID_CROSSWALK).astype("string")
    df.insert(0, "asof_date", asof_tag)
    df = df.drop_duplicates()
    if table_exists(path):
        DeltaTable(path).delete(f"asof_date = '{asof_tag}' AND source = '{df['source'].iloc[0]}'")
        write_deltalake(path, df, mode="append", schema_mode="merge")
    else:
        write_deltalake(path, df, mode="overwrite", schema_mode="overwrite")
    print(f"\nwrote {len(df)} identifier rows to factset.dim_account_map")
    print(df.to_string(index=False))
    return df

# --- vintage bundling ------------------------------------------------------
# asof_date (YYYYMMDD) is the VINTAGE key: every table written by either notebook in a
# quarter carries the same value, so the whole data pack is one auditable bundle. Power BI
# should read the latest COMPLETE vintage, which is what dim_vintage exposes — a vintage
# where PA landed but SPAR failed is present but incomplete, and consuming it would show
# holdings against last quarter's returns.
EXPECTED_VINTAGE_TABLES = (
    "spar_composite_returns",
    "pa_sector_weights",
    "pa_security_weights",
    "pa_characteristics",
)

def record_vintage(entries, notebook):
    """Append manifest rows, then rebuild dim_vintage from the full manifest.

    Reads asof_tag / TABLE_VINTAGE_MANIFEST / TABLE_DIM_VINTAGE from the notebook globals at
    call time — they are set in Cells 3 and 4b, which run before this is ever invoked.

    entries: {table_name: row_count}. Rebuilding from the manifest rather than tracking
    state means whichever notebook runs last produces the correct completeness, with no
    ordering assumption beyond both having run.
    """
    now = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")
    rows = [{"asof_date": asof_tag, "table_name": t, "row_count": int(n),
             "notebook": notebook, "written_utc": now} for t, n in entries.items()]
    manifest_add = pd.DataFrame(rows).astype("string")
    manifest_add["row_count"] = [int(n) for n in entries.values()]

    if table_exists(TABLE_VINTAGE_MANIFEST):
        # Replace this notebook's rows for this vintage so a re-run does not duplicate.
        DeltaTable(TABLE_VINTAGE_MANIFEST).delete(
            f"asof_date = '{asof_tag}' AND notebook = '{notebook}'")
        write_deltalake(TABLE_VINTAGE_MANIFEST, manifest_add, mode="append",
                        schema_mode="merge")
        manifest = DeltaTable(TABLE_VINTAGE_MANIFEST).to_pandas()
    else:
        write_deltalake(TABLE_VINTAGE_MANIFEST, manifest_add, mode="overwrite",
                        schema_mode="overwrite")
        manifest = manifest_add.copy()

    present = manifest.groupby("asof_date")["table_name"].agg(lambda s: sorted(set(s)))
    dim = pd.DataFrame({
        "asof_date": present.index,
        "tables_present": [",".join(v) for v in present],
        "n_tables": [len(v) for v in present],
        "is_complete": [set(EXPECTED_VINTAGE_TABLES) <= set(v) for v in present],
    }).sort_values("asof_date")
    dim["asof_date_iso"] = pd.to_datetime(dim["asof_date"], format="%Y%m%d").dt.date
    complete = dim.loc[dim["is_complete"], "asof_date"]
    latest_complete = complete.max() if len(complete) else None
    dim["is_latest_complete"] = dim["asof_date"] == latest_complete
    dim["is_latest"] = dim["asof_date"] == dim["asof_date"].max()

    write_deltalake(TABLE_DIM_VINTAGE, dim, mode="overwrite", schema_mode="overwrite")
    print(f"\nvintage {asof_tag}: {', '.join(f'{t}={n}' for t, n in entries.items())}")
    print(dim.to_string(index=False))
    if latest_complete is None:
        print("*** no vintage is complete yet — expected "
              f"{list(EXPECTED_VINTAGE_TABLES)}. Run the other notebook before pointing")
        print("*** Power BI at this data.")
    elif latest_complete != asof_tag:
        print(f"*** this vintage ({asof_tag}) is NOT yet complete; latest complete is "
              f"{latest_complete}. Power BI should stay on {latest_complete}.")
    return dim


In [ ]:
# === Cell 3: THE CONFIG BLOCK ==============================================

CURRENCY = "USD"
AS_OF_RELATIVE = "0CQ"       # sent as-is; Cell 4b resolves it for labelling
FREQUENCY = "Single"         # point-in-time snapshot, not a series

# No startdate is sent. PADateParameters.startdate is optional in 4.0.0, and at Single
# frequency a start date has nothing to do — the calculation is one observation at enddate.
# Omitting it keeps the request minimal and dynamic.
SEND_ABSOLUTE_END_DATE = False   # True for a backfill: pins the request to one quarter

# --- date probe: hardened ids ----------------------------------------------
# convert_pa_dates_to_absolute_format requires enddate + componentid + account. Pin a BASIC
# WEIGHTS component and pass a SINGLE account — the smallest, fastest call that satisfies
# the endpoint. Hardcoded on purpose: a pinned id needs no lookup, so the date resolves
# without waiting on component-name resolution.
DATE_PROBE_COMPONENT = "<TODO pinned basic-weights component id>"
DATE_PROBE_ACCOUNT = "<TODO one PA holdings account, path.ACCT>"

# --- the document (the one thing that must be right) -----------------------
PA_DOCUMENT = "<TODO PA3 document path>"

TILES = {
    "weights": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "componentdetail": "GROUPSALL",   # group rows AND security rows in one response
        "split_grain": True,
    },
    # Characteristics are published at TOTAL grain only, already aggregated by the engine —
    # one row per portfolio, not per sector. So componentdetail is TOTALS and there is no
    # grain to split; strategy_code alone organises the table.
    "characteristics": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "componentdetail": "TOTALS",
        "split_grain": False,
    },
}

# --- accounts --------------------------------------------------------------
# PA needs the HOLDINGS account path, not the returns ACCT the SPAR notebook uses — they
# are different objects, so do not copy one across.
# holdingsmode: B&H, TBR, OMS, EXT or VLT. Cell 4c reads these off the saved component.
STRATEGIES = {
    "LC":   {"label": "Large Cap",           "acct": "<TODO>", "holdingsmode": "B&H"},
    "SMID": {"label": "SMID",                "acct": "<TODO>", "holdingsmode": "B&H"},
    "LCS":  {"label": "Large Cap Select",    "acct": "<TODO>", "holdingsmode": "B&H"},
    "CONC": {"label": "Concentrated Equity", "acct": "<TODO>", "holdingsmode": "B&H"},
}

# --- call shape ------------------------------------------------------------
# False: one unit per tile with all four accounts, `benchmarks` OMITTED so each account
#        uses its saved default benchmark. 2 units. Strategy and benchmark come from the
#        response rather than the unit key.
# True:  one unit per port+bench pair, explicit single benchmark. 8 units. Strategy and
#        benchmark come from the unit key — easier to debug if attribution misbehaves,
#        but needs BENCHMARK_GROUPS filled in.
PER_PAIR_UNITS = False

# --- benchmarks: iShares ETF proxies, NOT the official indices -------------
# PA is holdings-based, so its benchmark needs CONSTITUENTS — and HBCM is not entitled to
# official Russell constituent data. The tracking ETF is the proxy: it holds the index and
# its holdings are available.
#
# The SPAR notebook uses the OFFICIAL index return streams, because returns-based analysis
# needs only a return series and has no entitlement obstacle.
#
# >>> PA and SPAR are therefore measured against DIFFERENT benchmarks. An ETF differs from
# >>> its index by expense ratio, cash drag, sampling and timing, so PA active weights will
# >>> not tie exactly to SPAR relative returns. Expected, not an error — but the two must
# >>> never be presented as though they shared a benchmark.
#
# Expected mapping (Cell 4c checks it against what the component actually has saved):
#   r1000  Russell 1000  -> IWB  iShares Russell 1000 ETF
#   r2500  Russell 2500  -> SMMD iShares Russell 2500 ETF
#   r3000  Russell 3000  -> IWV  iShares Russell 3000 ETF
# Tickers are named for identification only; the FactSet identifier and prefix need
# verifying, and an ETF used as a PA benchmark may need a holdings-capable prefix.
BENCHMARK_GROUPS = {
    "r1000": {"label": "iShares Russell 1000 ETF", "ticker": "IWB",
              "index": "Russell 1000", "id": "<TODO>"},   # LC + LCS
    "r2500": {"label": "iShares Russell 2500 ETF", "ticker": "SMMD",
              "index": "Russell 2500", "id": "<TODO>"},   # SMID
    "r3000": {"label": "iShares Russell 3000 ETF", "ticker": "IWV",
              "index": "Russell 3000", "id": "<TODO>"},   # CONC
}
BENCH_GROUP_OF = {"LC": "r1000", "LCS": "r1000", "SMID": "r2500", "CONC": "r3000"}
# The failure mode worth catching: a component silently pointed at the official index, which
# either 403s on entitlement or returns no constituents.
VERIFY_SAVED_BENCHMARKS = True

# --- security-grain filtering ---------------------------------------------
# Drop security rows with no portfolio weight — benchmark constituents not held. An R3000
# benchmark otherwise contributes thousands of rows no top-N view wants.
# Cost: loses security-level active weight for names you don't own, so
# largest-underweight-not-held cannot be read off pa_security_weights. Sector grain is
# unaffected. Harmless no-op if the component already hides them.
HIDE_BENCH_ONLY_SECURITIES = True

# --- output column mapping -------------------------------------------------
# STACH labels come from the component, so they can't be known in advance. First candidate
# present wins and is renamed to the canonical key. Cell 7 reports what actually arrived.
# FSYM comes in several flavours and PA can expose more than one. Each is canonicalised
# separately rather than collapsed, because they identify different things: the security, the
# regional listing, and the issuer/entity. Keeping them apart is what lets a security
# dimension join on the right one later.
COLUMN_HINTS = {
    "fsym_perm_id": ["fsym_perm_id", "fsym_security_id", "fsym_id", "perm_id"],
    "fsym_regional_id": ["fsym_regional_id", "fsym_listing_id", "regional_id"],
    "fsym_entity_id": ["fsym_entity_id", "entity_id", "fsym_company_id"],
    "ticker": ["ticker", "ticker_symbol", "ticker_exchange", "symbol", "local_symbol"],
    "security_name": ["security_name", "security", "name", "company_name",
                      "description", "issue_name"],
    # Only if the component exposes it as a real column; otherwise the sector is derived
    # from the STACH grouping, since FactSet does not repeat it on every security row.
    "gics_sector": ["gics_sector", "sector"],
    "cash_flag": ["cash_flag", "is_cash", "cash", "asset_class", "security_type"],
    "ultimate_parent_fsym_id": ["ultimate_parent_fsym_id", "fsym_ultimate_parent_id",
                                "ult_parent_fsym_id", "ultimate_parent_id"],
    # PA's precalculated weights — authoritative at every grain. Never re-derive.
    "port_weight": ["port_weight", "portfolio_weight", "port._weight", "weight_port",
                    "portfolio_ending_weight"],
    "bench_weight": ["bench_weight", "benchmark_weight", "bench._weight", "weight_bench"],
    "active_weight": ["active_weight", "active_wt", "weight_active", "difference"],
    # Whatever PA echoes for the account — an Orion id, a path.ACCT form, or a name.
    "account_out": ["account", "portfolio", "account_name", "portfolio_name", "port",
                    "account_id", "acct"],
    "benchmark_out": ["benchmark", "benchmark_id", "bench", "benchmark_symbol"],
    "benchmark_name_out": ["benchmark_name", "bench_name", "benchmark_description"],
}
# fsym_perm_id and a weight are non-negotiable at security grain; ticker and name are
# strongly wanted but a missing one should not block a quarter's load.
REQUIRED_CANONICAL = ["fsym_perm_id", "port_weight"]
WANTED_CANONICAL = ["ticker", "security_name", "fsym_regional_id", "fsym_entity_id",
                    "ultimate_parent_fsym_id", "cash_flag"]

# --- OneLake target --------------------------------------------------------
WORKSPACE_ID = "1b9fac18-9d75-4437-ab6c-b6ba44ff46a8"   # HBCM - Production
LAKEHOUSE_ID = "7cdf13b1-4586-4a02-b8ff-72fcf6db1277"   # hbcm_datahub
# NOTE the `.Lakehouse` suffix on the item id — required in a OneLake ABFSS path.
# Without it the write does not land in the lakehouse's managed area, so the table
# never registers and never appears in the SQL endpoint or Power BI.
ONELAKE = (f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/"
           f"{LAKEHOUSE_ID}.Lakehouse")
RAW_DIR = f"{ONELAKE}/Files/raw/pa"
TABLE_SECTOR = f"{ONELAKE}/Tables/factset/pa_sector_weights"
TABLE_SECURITY = f"{ONELAKE}/Tables/factset/pa_security_weights"
TABLE_CHARACTERISTICS = f"{ONELAKE}/Tables/factset/pa_characteristics"
TABLE_RUN_LOG = f"{ONELAKE}/Tables/factset/factset_run_log"
TABLE_VINTAGE_MANIFEST = f"{ONELAKE}/Tables/factset/vintage_manifest"
TABLE_DIM_VINTAGE = f"{ONELAKE}/Tables/factset/dim_vintage"
TABLE_DIM_ACCOUNT_MAP = f"{ONELAKE}/Tables/factset/dim_account_map"
TABLE_DIM_STRATEGY = f"{ONELAKE}/Tables/factset/dim_strategy"

assert tuple(STRATEGIES) == STRATEGY_CODES, (
    f"STRATEGIES keys {tuple(STRATEGIES)} must match the canonical STRATEGY_CODES "
    f"{STRATEGY_CODES} — this is the join key for every table in the model"
)
for _c, _s in STRATEGIES.items():
    assert _s["label"] == STRATEGY_LABELS[_c], f"{_c}: label disagrees with STRATEGY_LABELS"

# Derived, not hand-maintained: adding a strategy above is enough.
# Keys are matched against the account value in the response, trimmed both sides.
ACCT_TO_CODE = {str(s["acct"]).strip(): c for c, s in STRATEGIES.items()}
print(f"{len(TILES)} tiles, {len(STRATEGIES)} strategies, "
      f"{'per-pair (8 units)' if PER_PAIR_UNITS else 'multi-port (1 unit per tile)'}")

In [ ]:
# === Cell 4: resolve component ids by name — every run ====================
# Ids are not stable: re-saving a component can mint a new one, and a stale id 400s with
# nothing pointing at the id as the cause. The workstation NAME is the contract.

def resolve_component_ids(document=PA_DOCUMENT):
    summary, headers = call_with_headers(comp_api.get_pa_components, document=document)
    RUN_META.setdefault("headers", {}).update(headers)

    by_name = {}
    for cid, meta in (summary.data or {}).items():
        by_name.setdefault(_field(meta, "name"), []).append(cid)

    resolved, drift, missing = {}, [], []
    for tile, cfg in TILES.items():
        hits = by_name.get(cfg["component_name"], [])
        if len(hits) != 1:
            missing.append((tile, cfg["component_name"], len(hits)))
            continue
        resolved[tile] = hits[0]
        if cfg.get("pinned_componentid") and cfg["pinned_componentid"] != hits[0]:
            drift.append((tile, cfg["component_name"], cfg["pinned_componentid"], hits[0]))
    return resolved, drift, missing, by_name

RESOLVED_COMPONENTS, COMPONENT_DRIFT, COMPONENT_MISSING, COMPONENTS_BY_NAME = \
    resolve_component_ids()

for tile, cid in RESOLVED_COMPONENTS.items():
    print(f"{tile:<18} {cid}  ({TILES[tile]['component_name']})")

if COMPONENT_DRIFT:
    print("\n*** COMPONENT ID DRIFT — the component was re-saved. Confirm its columns")
    print("*** still match, then update pinned_componentid:")
    for tile, name, was, now in COMPONENT_DRIFT:
        print(f"    {tile}: {name!r}  {was} -> {now}")

if COMPONENT_MISSING:
    print("\nUnresolved tiles:", COMPONENT_MISSING)
    print("Available component names in", PA_DOCUMENT)
    for name, cids in sorted(COMPONENTS_BY_NAME.items(), key=lambda kv: str(kv[0])):
        print(f"    {name!r}: {cids}")

assert not COMPONENT_MISSING, "every tile needs exactly one matching component name"
RUN_META["components"] = dict(RESOLVED_COMPONENTS)
RUN_META["component_drift"] = [list(d) for d in COMPONENT_DRIFT]

In [ ]:
# === Cell 4b: resolve 0CQ to an absolute date — the authority =============
# PA has a DatesApi; SPAR does not. Whatever this returns is what gets SENT and what LABELS
# every row, in this notebook and in the SPAR one — so there is a single as-of date across
# both pipelines and no locally computed guess in the data.
#
# enddate, componentid and account are all REQUIRED (only startdate and calendar are
# optional). The component is PINNED rather than resolved by name, so this no longer depends
# on Cell 4 and could run first.

d_api = dates_api.DatesApi(api_client)

def _fallback_quarter_end(today=None):
    """Last COMPLETED calendar quarter end. Used only if the API call fails."""
    d = today or dt.date.today()
    qe = dt.date(d.year, ((d.month - 1) // 3) * 3 + 1, 1) - dt.timedelta(days=1)
    return qe.strftime("%Y%m%d")

def resolve_as_of(relative=AS_OF_RELATIVE):
    """(absolute YYYYMMDD, source) for `relative`, per PA.

    Pinned basic-weights component + a single account, no startdate: the smallest round trip
    that satisfies the endpoint's required args, and it needs no component lookup.
    """
    if "TODO" in DATE_PROBE_COMPONENT or "TODO" in DATE_PROBE_ACCOUNT:
        return _fallback_quarter_end(), "local fallback (DATE_PROBE_* not set)"
    try:
        resp, headers = call_with_headers(
            d_api.convert_pa_dates_to_absolute_format,
            enddate=relative, componentid=DATE_PROBE_COMPONENT,
            account=DATE_PROBE_ACCOUNT,
        )
        RUN_META.setdefault("headers", {}).update(headers)
        end = _field(resp.data, "enddate")
        start = _field(resp.data, "startdate")
        if end:
            return str(end), f"PA DatesApi (startdate={start})"
    except Exception as e:
        print(f"date conversion failed: {e!r}")
    return _fallback_quarter_end(), "local fallback (PA DatesApi unavailable)"

AS_OF_ABS, AS_OF_SOURCE = resolve_as_of()
# SENT stays dynamic so the scheduled job needs no maintenance; LABELLED is the resolved
# absolute date, so the data carries a real one.
AS_OF = AS_OF_ABS if SEND_ABSOLUTE_END_DATE else AS_OF_RELATIVE
asof_tag = AS_OF_ABS

_local = _fallback_quarter_end()
print(f"{AS_OF_RELATIVE} -> {AS_OF_ABS}   source: {AS_OF_SOURCE}")
print(f"enddate SENT: {AS_OF!r}   asof_date LABELLED: {asof_tag}   (no startdate)")
if AS_OF_ABS != _local:
    # Not necessarily wrong — PA may use a trading-calendar quarter end, or 0CQ may mean
    # the in-progress quarter. But it is worth knowing they differ.
    print(f"NOTE: differs from the locally computed quarter end ({_local}). "
          f"PA's answer is used.")

RUN_META.update({
    "asof_relative": AS_OF_RELATIVE,
    "asof_absolute": AS_OF_ABS,
    "asof_source": AS_OF_SOURCE,
    "asof_local_computed": _local,
    "frequency": FREQUENCY,
    "currency": CURRENCY,
    "end_date_sent": AS_OF,
    "send_absolute_end_date": SEND_ABSOLUTE_END_DATE,
})

# Hand the resolved date to the SPAR notebook so both pipelines share one as-of.
# SPAR reads this if present and falls back to calling PA itself.
try:
    notebookutils.fs.put(f"{ONELAKE}/Files/raw/_asof/{AS_OF_RELATIVE}.json",
                         json.dumps({"relative": AS_OF_RELATIVE, "absolute": AS_OF_ABS,
                                     "source": AS_OF_SOURCE,
                                     "resolved_utc": RUN_META["run_started_utc"]}), True)
except Exception as e:
    print(f"could not publish resolved as-of ({e!r}) — SPAR will resolve it itself")

In [ ]:
# === Cell 4c: read the saved config off each component ====================
# PAComponent exposes the accounts, benchmarks, currency, dates and snapshot flag SAVED IN
# THE DOCUMENT — so the document path plus two component names is enough to fill in account
# paths, holdings modes and benchmark ids. Run once, paste into Cell 3, then skip.

COMPONENT_META = {}
for tile, cid in RESOLVED_COMPONENTS.items():
    try:
        comp = comp_api.get_pa_component_by_id(id=cid)
    except fds.sdk.PAEngine.ApiException as e:
        print(f"{tile}: lookup failed {e.status} {e.body}")
        continue
    d = comp.data
    accts = [{"id": _field(a, "id"), "holdingsmode": _field(a, "holdingsmode")}
             for a in (_field(d, "accounts") or [])]
    benches = [{"id": _field(b, "id"), "holdingsmode": _field(b, "holdingsmode")}
               for b in (_field(d, "benchmarks") or [])]
    COMPONENT_META[tile] = {
        "componentid": cid,
        "name": _field(d, "name"),
        "category": _field(d, "category"),
        "path": _field(d, "path"),
        "currency": _field(d, "currencyisocode"),
        # snapshot=True => point-in-time, which is what a 0CQ weights pull wants. If it is
        # False the component is a subperiod calculation and "Single" may not mean holdings
        # AS OF the quarter end.
        "snapshot": _field(d, "snapshot"),
        "saved_dates": str(_field(d, "dates")),
        "saved_accounts": accts,
        "saved_benchmarks": benches,
    }
    print(f"\n=== {tile} ({COMPONENT_META[tile]['name']}) ===")
    for k in ("path", "category", "currency", "snapshot", "saved_dates"):
        print(f"  {k:<16} {COMPONENT_META[tile][k]}")
    for a in accts:
        print(f"  ACCOUNT          id={a['id']!r} holdingsmode={a['holdingsmode']!r}")
    for b in benches:
        print(f"  BENCHMARK        id={b['id']!r}")

RUN_META["component_meta"] = COMPONENT_META

# Verify the saved benchmark is the iShares ETF proxy, not the official index. A component
# pointed at the index either 403s on entitlement or returns no constituents — and either
# way the weights would be wrong or absent rather than obviously broken.
if VERIFY_SAVED_BENCHMARKS:
    expected = {g["ticker"].upper(): g for g in BENCHMARK_GROUPS.values()}
    expected_idx = {g["index"].upper(): g for g in BENCHMARK_GROUPS.values()}
    print("\nbenchmark check (expecting iShares ETF proxies, not official indices):")
    for tile, meta in COMPONENT_META.items():
        for b in meta["saved_benchmarks"]:
            bid = str(b["id"] or "")
            up = bid.upper()
            hit_etf = next((t for t in expected if t in up), None)
            hit_idx = next((ix for ix in expected_idx if ix.replace(" ", "") in
                            up.replace(" ", "")), None)
            if hit_etf:
                verdict = f"OK  ETF proxy ({hit_etf})"
            elif hit_idx:
                verdict = (f"*** OFFICIAL INDEX ({hit_idx}) — PA needs constituents and "
                           f"HBCM is not entitled; point this at the ETF")
            else:
                verdict = "?   unrecognised — confirm manually"
            print(f"  {tile:<16} {bid:<28} {verdict}")
print("\nMap each ACCOUNT id into Cell 3's STRATEGIES, then set")
print("ACCT_TO_CODE = {s['acct']: c for c, s in STRATEGIES.items()}")

In [ ]:
# === Cell 4d: DISCOVERY — optional lookups ================================
a_api = accounts_api.AccountsApi(api_client)
col_api = columns_api.ColumnsApi(api_client)
grp_api = groups_api.GroupsApi(api_client)
frq_api = frequencies_api.FrequenciesApi(api_client)

# GROUPSALL sets the ROW GRAIN, not which columns exist. If precalculated port/bench/active
# weight, FSYM perm id, cash flag or ultimate parent are missing from the output, the
# component doesn't expose them — edit it, or override PACalculationParameters.columns.
#   print(col_api.get_pa_columns(name="weight", category="", directory=""))
#   print(col_api.get_pa_columns(name="fsym", category="", directory=""))

# Confirm the saved grouping really is GICS sector — Cell 7 projects it onto every holding
# and is blind to what it represents.
#   print(grp_api.get_pa_groups())

#   print(frq_api.get_pa_frequencies())        # confirm "Single"
#   print(a_api.get_accounts(path="Client:/")) # browse holdings accounts
print("uncomment the lookup you need")

In [ ]:
# === Cell 5: build units ==================================================
# Default: one unit per tile, all four accounts, `benchmarks` OMITTED so each account uses
# its saved default. That is what lets four composites with three benchmarks share a unit.

def pa_dates() -> PADateParameters:
    # No startdate: optional in 4.0.0, and meaningless at Single frequency where the
    # calculation is one observation at enddate.
    # Keyword args throughout — 4.0.0 dropped `required` on enddate/frequency, which
    # reshuffled positional arguments across the PA calculation endpoints.
    return PADateParameters(enddate=AS_OF, frequency=FREQUENCY)

def _accounts(codes):
    return [PAIdentifier(id=STRATEGIES[c]["acct"],
                         holdingsmode=STRATEGIES[c]["holdingsmode"]) for c in codes]

def build_multiport_unit(tile_name, tile_cfg) -> PACalculationParameters:
    # `benchmarks` deliberately not passed — omitting it uses each account's default.
    return PACalculationParameters(
        componentid=RESOLVED_COMPONENTS[tile_name],
        accounts=_accounts(list(STRATEGIES)),
        dates=pa_dates(),
        currencyisocode=CURRENCY,
        componentdetail=tile_cfg["componentdetail"],
    )

def build_pair_unit(tile_name, tile_cfg, code) -> PACalculationParameters:
    bmk = BENCHMARK_GROUPS[BENCH_GROUP_OF[code]]
    return PACalculationParameters(
        componentid=RESOLVED_COMPONENTS[tile_name],
        accounts=_accounts([code]),
        benchmarks=[PAIdentifier(id=bmk["id"])],
        dates=pa_dates(),
        currencyisocode=CURRENCY,
        componentdetail=tile_cfg["componentdetail"],
    )

UNIT_KEYS, units = {}, {}
for tile_name, tile_cfg in TILES.items():
    if PER_PAIR_UNITS:
        for code in STRATEGIES:
            key = f"{tile_name}__{code}"
            units[key] = build_pair_unit(tile_name, tile_cfg, code)
            UNIT_KEYS[key] = (tile_name, code)
    else:
        units[tile_name] = build_multiport_unit(tile_name, tile_cfg)
        UNIT_KEYS[tile_name] = (tile_name, None)   # strategy comes from the response

params_root = PACalculationParametersRoot(
    data=units,
    meta=CalculationMeta(
        contentorganization="SimplifiedRow",
        stach_content_organization="SimplifiedRow",
        contenttype="Json",
        format="JsonStach",
    ),
)
RUN_META["unit_keys"] = list(units)
RUN_META["per_pair_units"] = PER_PAIR_UNITS
for key, (tile_name, code) in UNIT_KEYS.items():
    who = code or ", ".join(STRATEGIES)
    print(f"{key:<26} {TILES[tile_name]['componentdetail']:<10} {who}")

In [ ]:
# === Cell 6: submit + poll ================================================

def run_pa(params_root, deadline=10, poll_interval=3, timeout=900):
    wrapper, headers = call_with_headers(
        calc_api.post_and_calculate,
        x_fact_set_api_long_running_deadline=deadline,
        pa_calculation_parameters_root=params_root,
    )
    RUN_META.setdefault("headers", {}).update(headers)

    code = wrapper.get_status_code()
    if code == 200:
        status_root = wrapper.get_response_200()
    elif code == 201:
        status_root = wrapper.get_response_201()
    elif code == 202:
        status_root = wrapper.get_response_202()
        calc_id = status_root.data.calculationid
        deadline_at = time.time() + timeout
        while True:
            if time.time() > deadline_at:
                calc_api.cancel_calculation_by_id(id=calc_id)
                raise TimeoutError(f"calc {calc_id} exceeded {timeout}s (cancelled)")
            poll = calc_api.get_calculation_status_by_id(id=calc_id)
            if poll.get_status_code() == 200:
                status_root = poll.get_response_200()
                break
            if poll.get_status_code() != 202:
                raise RuntimeError(f"unexpected poll status {poll.get_status_code()}")
            time.sleep(poll_interval)
    else:
        raise RuntimeError(f"unexpected submit status {code}")

    calc_id = status_root.data.calculationid
    out, unit_meta = [], {}
    for unit_id, unit_status in (status_root.data.units or {}).items():
        st = _field(unit_status, "status")
        # Whatever the engine reports per unit — errors, progress, timing — is worth
        # keeping; it is the only record of why a unit came back empty.
        unit_meta[unit_id] = {
            "status": st,
            "error": str(_field(unit_status, "error") or ""),
            "progress": str(_field(unit_status, "progress") or ""),
        }
        if st != "Success":
            out.append((unit_id, None, st))
            continue
        res, rheaders = call_with_headers(
            calc_api.get_calculation_unit_result_by_id, id=calc_id, unit_id=unit_id)
        RUN_META.setdefault("headers", {}).update(rheaders)
        out.append((unit_id, res, st))
    return calc_id, out, unit_meta

calc_id, results, UNIT_META = run_pa(params_root)
RUN_META["calculation_id"] = calc_id
RUN_META["unit_status"] = UNIT_META

failed = [(u, s) for u, r, s in results if r is None]
print(f"calc={calc_id} ok={len(results) - len(failed)} failed={len(failed)}")
for u, s in failed:
    print(f"  FAILED {u}: {s} — {UNIT_META[u]['error']}")
print("\ncaptured headers:")
for k, v in RUN_META.get("headers", {}).items():
    print(f"    {k}: {v}")
assert results and not failed, "resolve failures before writing to the lakehouse"

In [ ]:
# === Cell 7: raw landing, STACH parse, sector projection ==================
for unit_id, res, _ in results:
    notebookutils.fs.put(f"{RAW_DIR}/asof={asof_tag}/{unit_id}.json",
                         json.dumps(res.to_dict(), default=str), True)
print(f"landed {len(results)} raw payloads under {RAW_DIR}/asof={asof_tag}/")

def norm(cols):
    return [str(c).strip().replace(" ", "_").replace("-", "_").lower() for c in cols]

GROUPING_COLS = ("gics_sector", "sector", "group", "grouping", "group_name", "group_1")

def first_present(df, candidates):
    return next((c for c in candidates if c in df.columns), None)



def derive_sector(df):
    """Classify each row's grain, then take the security's sector from the hierarchy.

    Under GROUPSALL the sector is the GROUP a security sits beneath, not a column on the
    security row — FactSet does not repeat it there. classify_grain uses the blank identifier
    on group and total rows plus STACH's group_level; assign_ancestors walks the level stack
    to give each security its ancestor labels exactly, including nesting deeper than one
    level. If the component does expose a real sector column, that wins.
    """
    grain, grep = classify_grain(df)
    out = assign_ancestors(df.assign(row_grain=grain),
                           label_cols=tuple(c for c in GROUPING_COLS if c in df.columns)
                                      + ("security_name",))
    if "gics_sector" in out.columns and not _blank(out["gics_sector"]).all():
        method = "column"          # component exposes it directly; leave it alone
    else:
        out["gics_sector"] = out["parent_group"]
        method = "hierarchy:parent_group"
    return out, method, grep

frames, sector_methods, collisions, schemas, undeclared = [], {}, set(), {}, set()
id_unmatched = set()
for unit_id, res, _ in results:
    tile_name, code = UNIT_KEYS[unit_id]
    for i, (tid, df, schema, levels) in enumerate(stach_parse(res)):
        df, tp = type_from_schema(df, schema)
        undeclared.update(tp["undeclared"])
        schemas[f"{unit_id}[{i}]"] = {"table_id": tid, **{k: v for k, v in tp.items() if v}}
        # STACH's declared hierarchy level, carried as a column so the grain split and the
        # sector projection both work off the engine's own structure.
        df["group_level"] = levels if len(levels) == len(df) else pd.NA
        df.columns = norm(df.columns)
        for canonical, candidates in COLUMN_HINTS.items():
            hit = first_present(df, [c for c in candidates if c != canonical])
            if canonical not in df.columns and hit:
                df = df.rename(columns={hit: canonical})

        if TILES[tile_name]["split_grain"]:
            df, method, grep = derive_sector(df)
            sector_methods[f"{unit_id}[{i}]"] = {"method": method, **grep}

        # Strategy: from the unit key in per-pair mode, otherwise matched from the
        # response's account column, since a multi-port unit holds all four. The response may
        # echo an Orion account id, a path.ACCT form, or a display name, so matching is
        # normalised rather than exact and every identifier seen is recorded.
        if code is not None:
            df["strategy_code"] = code
            df["strategy_matched_by"] = "unit key"
            for _acct_val in ({df.get("account_out").iloc[0]}
                              if "account_out" in df.columns and len(df) else {None}):
                record_identifiers(code, "pa", "account", sent=STRATEGIES[code]["acct"],
                                   observed_id=_acct_val, matched_by="unit key")
        elif "account_out" in df.columns:
            codes, how, unmatched = match_strategy(df["account_out"], ACCT_TO_CODE)
            df["strategy_code"] = codes
            df["strategy_matched_by"] = how
            id_unmatched.update(unmatched)
            for _obs, _c, _h in set(zip(df["account_out"].astype("string"),
                                        codes.astype("string"), how.astype("string"))):
                if pd.notna(_c):
                    record_identifiers(_c, "pa", "account",
                                       sent=STRATEGIES.get(_c, {}).get("acct"),
                                       observed_id=_obs, matched_by=_h)
        else:
            df["strategy_code"] = pd.NA
            df["strategy_matched_by"] = pd.NA
        df["strategy"] = df["strategy_code"].map(
            {c: s["label"] for c, s in STRATEGIES.items()})

        # Benchmark identity comes from the response when it was not sent (multi-port omits
        # `benchmarks` so each account uses its saved default). Capture symbol and name.
        _b_id = df["benchmark_out"] if "benchmark_out" in df.columns else pd.NA
        df["benchmark_out"] = _b_id
        _b_name = df["benchmark_name_out"] if "benchmark_name_out" in df.columns else pd.NA
        df["benchmark_name_out"] = _b_name
        if "benchmark_out" in df.columns and len(df):
            for _c, _bo, _bn in set(zip(df["strategy_code"].astype("string"),
                                        df["benchmark_out"].astype("string"),
                                        df["benchmark_name_out"].astype("string"))):
                if pd.notna(_c):
                    record_identifiers(_c, "pa", "benchmark",
                                       sent=(BENCHMARK_GROUPS[BENCH_GROUP_OF[_c]]["id"]
                                             if _c in BENCH_GROUP_OF else None),
                                       observed_id=_bo, observed_name=_bn,
                                       matched_by="from response")

        provenance = [
            ("asof_date",       asof_tag),
            ("asof_relative",   AS_OF_RELATIVE),
            ("tile",            tile_name),
            ("componentid",     RESOLVED_COMPONENTS[tile_name]),
            ("component_name",  TILES[tile_name]["component_name"]),
            ("componentdetail", TILES[tile_name]["componentdetail"]),
            ("currency",        CURRENCY),
            ("calculation_id",  calc_id),
            ("table_ix",        i),
            ("stach_table_id",  tid),
        ]
        df, clashed = dedupe_columns(df, {c for c, _ in provenance})
        if clashed:
            collisions.update(clashed)
        for pos, (col, val) in enumerate(provenance):
            df.insert(pos, col, val)
        frames.append(df)

tidy = pd.concat(frames, ignore_index=True)
if collisions:
    print(f"STACH columns renamed to avoid clashing with provenance: {sorted(collisions)}"
          f" (suffixed _src)")

print(f"\n{tidy.shape[0]} rows, {tidy.shape[1]} columns")
print("\ncolumns:", list(tidy.columns))
if undeclared:
    print(f"columns with no declared STACH type (fell back to sniffing): {sorted(undeclared)}")
print("\nsector projection method per unit/table:")
for k, v in sector_methods.items():
    print(f"    {k:<30} {v}")

print("\nrequested / required columns:")
for canonical in list(COLUMN_HINTS) + ["gics_sector"]:
    present = canonical in tidy.columns
    filled = int(tidy[canonical].notna().sum()) if present else 0
    flag = "" if present else "   <-- MISSING, add the real name to COLUMN_HINTS"
    print(f"    {canonical:<26} present={present} non_null={filled}{flag}")

print(f"\nstrategy_code resolved: {int(tidy['strategy_code'].notna().sum())} of {len(tidy)}")
if tidy["strategy_code"].isna().any() and not PER_PAIR_UNITS:
    seen = tidy.get("account_out")
    print("  unmapped account values:",
          sorted(set(seen.dropna().unique())) if seen is not None else "(no account column)")
    print("  -> set ACCT_TO_CODE in Cell 3 to map these, or use PER_PAIR_UNITS = True")
if id_unmatched:
    print(f"\nunmatched account identifiers from the response: {sorted(id_unmatched)}")
    print("  -> add the real value as a key in ACCT_TO_CODE; matching already tries exact,")
    print("  -> case-insensitive and normalised (prefix and .ACCT/.ACTM stripped) forms")

_w = tidy[tidy["tile"] == "weights"]
if len(_w) and "row_grain" in _w.columns:
    print("\nweights row grains (from the blank identifier + STACH group_level):")
    print(_w["row_grain"].value_counts().to_string())
    _sec = _w[_w["row_grain"] == "security"]
    for _c in ("ticker", "security_name", "fsym_perm_id", "fsym_regional_id",
               "fsym_entity_id", "ultimate_parent_fsym_id", "cash_flag"):
        if _c in _sec.columns:
            print(f"  {_c:<26} populated on {int(_sec[_c].notna().sum())} of "
                  f"{len(_sec)} security rows")
    if "gics_sector" in _sec.columns:
        print("\nholdings per derived sector (sanity-check against the workstation):")
        print(_sec.groupby(["strategy_code", "gics_sector"], dropna=False).size())
    _lvls = [c for c in _w.columns if c.startswith("group_l")]
    if _lvls:
        print(f"\nhierarchy levels reconstructed: {_lvls} (+ parent_group)")
else:
    print("\n*** no row_grain on the weights output — the grain split cannot run")
RUN_META["stach_schemas"] = schemas
RUN_META["undeclared_types"] = sorted(undeclared)
display(tidy.head(30))

In [ ]:
# === Cell 8: split by grain, drop benchmark-only securities, write ========
# Guarded rather than commented out: these asserts fail loudly until the grain
# discriminator, column names and strategy attribution are confirmed. A mixed-grain table
# double-counts silently and a sector-less holdings table looks fine.

weights = tidy[tidy["tile"] == "weights"].copy()
chars = tidy[tidy["tile"] == "characteristics"].copy()

assert "row_grain" in weights.columns, (
    "row_grain absent — Cell 7's derive_sector did not run, so the grains cannot be split"
)
print("\nrow grains in the weights output:")
print(weights["row_grain"].value_counts().to_string())
for canonical in REQUIRED_CANONICAL:
    assert canonical in weights.columns, (
        f"{canonical} absent — add its real STACH name to COLUMN_HINTS, or the component "
        f"does not expose it"
    )
_missing_wanted = [c for c in WANTED_CANONICAL if c not in weights.columns]
if _missing_wanted:
    print(f"NOTE not exposed by the component (not blocking): {_missing_wanted}")
    print("  -> add the real STACH name to COLUMN_HINTS, or add the column to the component")
assert tidy["strategy_code"].notna().all(), (
    "some rows have no strategy — fill ACCT_TO_CODE from Cell 7's unmapped values, or set "
    "PER_PAIR_UNITS = True so the strategy comes from the unit key"
)

# Two grains land, not three. The portfolio TOTAL row is dropped: it carries no information
# the other grains lack, and its only real use is the 100% check below — which the security
# and sector rows already answer. Characteristics is where total grain lives, and it comes
# from its own TOTALS component.
n_total_rows = int((weights["row_grain"] == "total").sum())
sectors = weights[weights["row_grain"] == "group"].copy()
securities = weights[weights["row_grain"] == "security"].copy()
print(f"dropped {n_total_rows} portfolio-total row(s) from the weights output "
      f"(total grain lives in pa_characteristics)")
RUN_META["weights_total_rows_dropped"] = n_total_rows
assert securities["gics_sector"].notna().all(), (
    "some holdings have no GICS sector — the grouping projection did not cover every "
    "security row; check the method Cell 7 reported"
)

# Benchmark-only securities: present in the index, not held. Dropped at security grain
# only — sector grain keeps the full picture, including active weight.
n_before = len(securities)
if HIDE_BENCH_ONLY_SECURITIES:
    pw = pd.to_numeric(securities["port_weight"], errors="coerce")
    securities = securities[pw.notna() & (pw != 0)]
n_dropped = n_before - len(securities)
print(f"security rows: {n_before} -> {len(securities)} "
      f"({n_dropped} benchmark-only dropped, HIDE_BENCH_ONLY_SECURITIES={HIDE_BENCH_ONLY_SECURITIES})")
RUN_META["security_rows_in"] = n_before
RUN_META["security_rows_kept"] = len(securities)
RUN_META["bench_only_dropped"] = n_dropped
RUN_META["sector_rows"] = len(sectors)
RUN_META["sector_methods"] = sector_methods

# PRINTED ONLY, NEVER WRITTEN. PA's precalculated weights are authoritative at both grains,
# so this sum exists to be looked at, not stored and not used to adjust anything. If the
# securities do not add to the sector row, the sector row still wins and the gap is PA
# methodology or the dropped benchmark-only names — it is not an error to reconcile away.
if "port_weight" in sectors.columns:
    _s = pd.to_numeric(sectors["port_weight"], errors="coerce").groupby(
        sectors["strategy_code"]).sum()
    _q = pd.to_numeric(securities["port_weight"], errors="coerce").groupby(
        securities["strategy_code"]).sum()
    print("\nport_weight by grain (informational — precalculated values are used as-is):")
    print(pd.DataFrame({"sector_grain": _s, "security_grain": _q}))

def write(path, df, name):
    if df.empty:
        print(f"skip {name}: no rows")
        return
    df = df.copy()
    df["asof_date_iso"] = pd.to_datetime(asof_tag, format="%Y%m%d").date()
    # Types were already set per table from its own STACH schema in Cell 7 — nothing to
    # re-infer here, and re-inferring would risk undoing a declared type.
    n_num = sum(1 for c in df.columns if str(df[c].dtype) in ("Float64", "Int64", "float64"))
    if table_exists(path):
        DeltaTable(path).delete(f"asof_date = '{asof_tag}'")
        mode = "append"
    else:
        mode = "overwrite"          # first write creates the table
    write_deltalake(path, df, mode=mode, schema_mode="merge")
    print(f"wrote {len(df)} rows to {name} (mode={mode}) — {n_num} numeric columns")
    if not n_num:
        print(f"  *** {name}: every column landed as text; Power BI would need a type")
        print(f"  *** conversion per measure. Check the component's declared column types.")

# --- validate join key and grain before anything is written --------------
print("\nvalidating join key and grain")
for _name, _df in (("pa_sector_weights", sectors),
                   ("pa_security_weights", securities),
                   ("pa_characteristics", chars)):
    if _df.empty:
        print(f"  NOTE {_name}: no rows")
        continue
    _codes = assert_strategy_key(_df, _name)
    print(f"  OK   {_name}: strategy_code canonical {_codes}")

assert_unique_grain(sectors, ["asof_date", "strategy_code", "gics_sector"],
                    "pa_sector_weights")
assert_unique_grain(securities, ["asof_date", "strategy_code", "fsym_perm_id"],
                    "pa_security_weights")
# Characteristics are TOTAL grain: exactly one row per strategy, already aggregated by the
# engine. More than one means the component is not really at TOTALS, or two accounts mapped
# to the same strategy_code.
if not chars.empty:
    assert_unique_grain(chars, ["asof_date", "strategy_code"], "pa_characteristics")
    _n_expected = chars["strategy_code"].nunique()
    print(f"  pa_characteristics: {len(chars)} rows for {_n_expected} strategies "
          f"({'one per strategy' if len(chars) == _n_expected else 'UNEXPECTED'})")

# --- the same security held in more than one strategy --------------------
# EXPECTED, not an error: LC and LCS are both large-cap, so they share names. What matters is
# the consequence — fsym_perm_id alone is NOT unique in this table, so a security dimension
# related on it will fan out and weights will sum across strategies unless strategy_code is
# in filter context. The grain assert above is what guarantees uniqueness on the FULL key.
if not securities.empty and "fsym_perm_id" in securities.columns:
    _held = (securities.groupby("fsym_perm_id")["strategy_code"]
                       .agg(n_strategies="nunique",
                            strategies=lambda s: "+".join(sorted(set(s)))))
    _shared = _held[_held["n_strategies"] > 1].sort_values("n_strategies", ascending=False)
    print(f"\ncross-strategy holdings: {len(_shared)} of {len(_held)} securities are held "
          f"by more than one strategy")
    if len(_shared):
        print(_shared["strategies"].value_counts().to_string())
        print("  -> expected. fsym_perm_id is therefore NOT a unique key here; any security")
        print("  -> dimension must be many-to-one and strategy_code must be in filter")
        print("  -> context before summing a weight, or it sums across strategies.")
    RUN_META["securities_total"] = int(len(_held))
    RUN_META["securities_multi_strategy"] = int(len(_shared))

    # Weights are the ONE place a column sum is meaningful, and only as a check that it
    # reaches 100% — never as a published figure. Both grains should independently total ~100
    # per strategy; a gap points at the grain split, the benchmark-only filter, or cash being
    # dropped for having no portfolio weight.
    _checks = {}
    for _label, _df in (("security", securities), ("sector", sectors)):
        if "port_weight" in _df.columns and not _df.empty:
            _checks[_label] = (pd.to_numeric(_df["port_weight"], errors="coerce")
                                 .groupby(_df["strategy_code"]).sum().round(2))
    if _checks:
        print("\nportfolio weight total per strategy — expect ~100 at BOTH grains:")
        print(pd.DataFrame(_checks).to_string())
        RUN_META["port_weight_totals"] = {
            g: {k: float(v) for k, v in s.items()} for g, s in _checks.items()}
        for _label, _s in _checks.items():
            _off = _s[(_s < 99) | (_s > 101)]
            if len(_off):
                print(f"  *** {_label} grain outside 99-101 for {list(_off.index)} — check the")
                print(f"  *** grain split, the benchmark-only filter, and whether cash is")
                print(f"  *** being dropped for having no portfolio weight")
        # The two grains describe the same portfolio, so they must agree with each other.
        if len(_checks) == 2:
            _d = (_checks["security"] - _checks["sector"]).abs().round(2)
            _bad = _d[_d > 0.5]
            if len(_bad):
                print(f"  *** security and sector totals disagree by >0.5 for "
                      f"{_bad.to_dict()} — the grain classification is suspect")

write(TABLE_SECTOR, sectors, "factset.pa_sector_weights")
write(TABLE_SECURITY, securities, "factset.pa_security_weights")
write(TABLE_CHARACTERISTICS, chars, "factset.pa_characteristics")

# The identifier crosswalk: every account and benchmark identifier observed, against
# strategy_code, so the semantic model joins on one key and a vendor changing what it echoes
# is visible rather than silently unmatched.
write_id_crosswalk(TABLE_DIM_ACCOUNT_MAP)

# --- dim_strategy: the dimension the semantic model filters on -----------
# Emitted here because this notebook runs first. One row per strategy, so the model has a
# real dimension to slice by instead of relying on strategy_code appearing consistently in
# every fact table.
dim_strategy = pd.DataFrame([
    {"strategy_code": c,
     "strategy": STRATEGY_LABELS[c],
     "pa_account": STRATEGIES[c]["acct"],
     "pa_holdingsmode": STRATEGIES[c]["holdingsmode"],
     "pa_benchmark_group": BENCH_GROUP_OF[c],
     "pa_benchmark_etf": BENCHMARK_GROUPS[BENCH_GROUP_OF[c]]["ticker"],
     "pa_benchmark_index": BENCHMARK_GROUPS[BENCH_GROUP_OF[c]]["index"],
     "sort_order": i}
    for i, c in enumerate(STRATEGY_CODES)
]).astype("string")
dim_strategy["sort_order"] = range(len(STRATEGY_CODES))
assert_unique_grain(dim_strategy, ["strategy_code"], "dim_strategy")
write_deltalake(TABLE_DIM_STRATEGY, dim_strategy, mode="overwrite", schema_mode="overwrite")
print(f"wrote {len(dim_strategy)} rows to factset.dim_strategy (full overwrite — it is a "
      f"small dimension, not history)")

In [ ]:
# === Cell 9: land the run log =============================================
# One row per run. Request keys are the difference between "we'll investigate" and "we know
# exactly what you sent" on a FactSet support ticket, and they only exist at call time.

RUN_META["run_finished_utc"] = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")

flat = {"asof_date": asof_tag}
for k, v in RUN_META.items():
    flat[k] = v if isinstance(v, (str, int, float, bool)) or v is None \
        else json.dumps(v, default=str)

run_log = pd.DataFrame([flat]).astype("string")
mode = "append" if table_exists(TABLE_RUN_LOG) else "overwrite"   # append-only history
write_deltalake(TABLE_RUN_LOG, run_log, mode=mode, schema_mode="merge")
print(f"logged run to factset.factset_run_log (mode={mode})")
for k in ("asof_absolute", "asof_source", "calculation_id", "bench_only_dropped"):
    print(f"    {k}: {RUN_META.get(k)}")

# Bundle this quarter into one auditable vintage and recompute which vintage is latest
# AND complete — that is the one Power BI should read.
record_vintage({"pa_sector_weights": len(sectors),
                "pa_security_weights": len(securities),
                "pa_characteristics": len(chars)}, "pa_weights_characteristics")

# Re-frame any Direct Lake semantic model BEFORE running VACUUM — vacuuming files a framed
# model still points at gives query errors on missing files. Order: write -> frame -> vacuum.

## To finish

1. `PA_DOCUMENT` + the two `component_name`s (Cell 3).
2. Run Cell 4c — it prints the accounts, holdings modes and benchmarks saved in each
   component. Map them into `STRATEGIES`, then set
   `ACCT_TO_CODE = {s["acct"]: c for c, s in STRATEGIES.items()}`.
3. Run to Cell 7 and read its reports: which columns arrived, which sector-projection path
   ran, how many rows resolved to a strategy, and holdings-per-sector counts.
4. Add real column names to `COLUMN_HINTS` until Cell 8's asserts pass.

If `ACCT_TO_CODE` can't be made to match the output's account values, set
`PER_PAIR_UNITS = True` — 8 units instead of 2, but strategy and benchmark come from the
unit key and nothing has to be matched.

## What to verify before trusting the numbers

- **The grain discriminator.** `group_mask()` guesses at a level/depth column, then a null
  security id. Both grains carry precalculated weights, so a wrong split double-counts.
- **The sector projection.** Cell 7 reports `direct:` or `ffill:`. `ffill` relies on STACH
  row order being hierarchical — cross-check holdings-per-sector against the workstation.
- **Active weight exists at both grains.** `GROUPSALL` sets the row grain, not the column
  set. If the component exposes only portfolio and benchmark weight, edit it or override
  `PACalculationParameters.columns`.
- **`snapshot` is True on the weights component** (Cell 4c). If False, it is a subperiod
  calculation and `Single` at `0CQ` may not mean holdings *as of* the quarter end.
- **The grouping really is GICS sector**, not another saved scheme (`get_pa_groups()`).
- **Cash.** Check whether cash lands inside a sector, in its own group, or with no sector,
  and whether `cash_flag` distinguishes it. Cash has no portfolio weight in some
  configurations, in which case `HIDE_BENCH_ONLY_SECURITIES` would drop it — the row
  counts printed in Cell 8 are the place to notice that.
- **PA's `0CQ`** may differ from the naive last-completed-quarter-end if it uses a trading
  calendar. Cell 4b prints both and prefers PA's.

## Consuming this from Power BI

**Three tables, three grains** — deliberately not one wide table, because PA supplies
precalculated weights at each grain and a shared table would double-count on any unfiltered
`SUM`:

| Table | Grain | Use |
|---|---|---|
| `factset.pa_sector_weights` | `asof_date` x `strategy_code` x sector | sector port / bench / active weights |
| `factset.pa_security_weights` | `asof_date` x `strategy_code` x `fsym_perm_id` | top-N holdings; held names only |
| `factset.pa_characteristics` | `asof_date` x `strategy_code` | portfolio characteristics, total grain — one row per strategy |

**Expected M:** `Lakehouse` connector → pick the table → nothing else. Weights are already
`Float64` and `asof_date_iso` is already a date.

**Joining:** `asof_date_iso` to a date dimension, `strategy_code` to a strategy dimension,
`fsym_perm_id` to a security dimension. `gics_sector` is already flattened onto each holding,
so a sector slicer works on `pa_security_weights` without touching `pa_sector_weights`.

`pa_characteristics` relates to `dim_strategy` one-to-one, so it needs no grain filter.

**Never sum across the two weight tables.** Use sector rows for sector totals and security
rows for holdings detail. They are not additive to each other.

**Benchmark caveat for any visual that shows both engines:** PA is measured against the
iShares ETF, SPAR against the official index. Don't put a PA active weight next to a SPAR
relative return and imply a common benchmark.

## Quarterly refresh, in order

Run **this notebook first** — it resolves `0CQ` and publishes the absolute date the SPAR
notebook reads, so both land on one as-of. Then SPAR, then re-frame the Direct Lake model,
then any `VACUUM` (write → frame → vacuum, never out of order).

Re-runnable: each write deletes the current `asof_date` before appending, so a repeat run
replaces the quarter instead of doubling it.

## Sources

Verified against **upstream `FactSet/enterprise-sdk` `main`** (PAEngine v3, SDK 4.0.0) —
`PACalculationParameters.md` (`accounts`/`benchmarks` as lists, `componentdetail` values,
`columns`, `groups`), `PAComponent.md` (saved `accounts`, `benchmarks`, `dates`,
`snapshot`), `PADateParameters.md`, `PAIdentifier.md`, `PACalculationsApi.md`,
`ComponentsApi.md`, `DatesApi.md` (`enddate` + `componentid` + `account` required;
`startdate`/`calendar` optional), `DateParametersSummary.md`, `ColumnsApi.md`,
`GroupsApi.md`, `FrequenciesApi.md`, and `BREAKING.md`.

Not against `code/python/PAEngine/v3/` in this repo, which is pinned at 2.2.2.